In [0]:
%pip install shap


In [0]:
# Restart Python Environment 
dbutils.library.restartPython()

In [0]:
#1 Verify SHAP Library
import shap

print("SHAP version:", shap.__version__)

In [0]:
#2 Load Production Model
import mlflow.pyfunc

purchase_model = mlflow.sklearn.load_model(
"models:/workspace.default.ecommerce_purchase_model@production"
)

In [0]:
# verify the model type 
print(type(purchase_model))

In [0]:
#3 Load the Feature Dataset,Load the Gold dataset again.
df = spark.read.table(
"workspace.ecommerce_lakehouse.gold_ml_training_dataset"
)

pdf = df.toPandas()

# Define features
feature_cols = [
"total_events",
"views",
"add_to_cart",
"purchases",
"avg_price",
"frequency",
"monetary_value",
"recency_days"
]

X = pdf[feature_cols]

In [0]:
#4 Create SHAP Explainer,,Because RandomForest is tree-based:
import shap
explainer = shap.TreeExplainer(purchase_model)

In [0]:
# verify dataset size
print("Dataset rows:", len(X))

In [0]:
#5 Compute SHAP Values .Use a subset (faster).
#use a dynamic sample size.

sample_size = min(200, len(X))
X_sample = X.sample(sample_size, random_state=42)
shap_values = explainer.shap_values(X_sample)

# This automatically adapts:
# If dataset < 200 → sample dataset size; If dataset > 200 → sample 200 rows


In [0]:
# 6  Visualize Feature Importance
shap.summary_plot(shap_values, X_sample)

In [0]:
#7 Save SHAP Visualization
import matplotlib.pyplot as plt

plt.savefig("/tmp/shap_summary.png")

In [0]:
#8 Log SHAP Artifact to MLflow,, Start a logginig ru “We also analyze feature impact using SHAP to understand what behaviors drive purchases in the model.” show the SHAP chart.
import mlflow

with mlflow.start_run(run_name="model_explainability"):

    mlflow.log_artifact("/tmp/shap_summary.png")